# 6. Two-Optimizer Adversarial cVAE + MMD Batch Correction

**Key differences from notebook 5 (single-optimizer GRL):**
- Two separate optimizers (encoder+decoder vs discriminator)
- 3 discriminator steps per 1 encoder step (standard GAN approach)
- Stronger discriminator (256->128, spectral normalization, LeakyReLU)
- MMD loss as additional/alternative regularizer
- `loss_type` in {'adversarial', 'mmd', 'both'}
- Full scIB metrics: kBET, iLISI, ASW_batch, cLISI, Sil_bio, NMI, ARI
- Comparison: Raw vs KL-cVAE vs Adv2 (adversarial / mmd / both)

In [ ]:
# Setup & Install
import subprocess, sys, os

if "google.colab" in sys.modules:
    repo = "/content/batchcor-rna-embeds"
    if not os.path.isdir(repo):
        subprocess.run(["git", "clone", "-b", "feat/scvi-batch-correction",
                        "https://github.com/onion-42/batchcor-rna-embeds.git", repo], check=True)
    else:
        subprocess.run(["git", "-C", repo, "pull", "origin", "feat/scvi-batch-correction"], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-e", "."], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q",
                     "scib-metrics", "leidenalg", "igraph"], check=True)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scanpy as sc
from pathlib import Path
from loguru import logger
from sklearn.metrics import silhouette_score

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.data_io import load_cohort
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL,
    COMPASS_PT_EMBEDDING_KEY,
    DATA_INTERIM_PT_DIR,
)
from batchcor_rna_emb.batch_correction.scvi_corrector import (
    CVAEAdv2Config, CVAEAdv2Corrector,
    CVAEConfig, CVAECorrector,
)
from batchcor_rna_emb.metrics.batch_metrics import (
    compute_kbet, compute_ilisi, compute_asw_batch, compute_graph_connectivity,
)
from batchcor_rna_emb.metrics.bio_metrics import (
    compute_clisi, compute_silhouette_bio, compute_nmi, compute_ari,
    run_leiden_clustering,
)
from batchcor_rna_emb.metrics.aggregation import geometric_mean, build_comparison_table

set_logger("INFO")
logger.info("Imports OK")

In [ ]:
EMB_KEY = COMPASS_PT_EMBEDDING_KEY
ADV2_KEY = "FM_COMPASS_PT_emb_Adv2_corrected"

if "google.colab" in __import__("sys").modules:
    DRIVE_DATA = Path("/content/drive/MyDrive/batchcor-rna-embeds/data")
    INTERIM_PT = DRIVE_DATA / "interim_PT"
else:
    INTERIM_PT = DATA_INTERIM_PT_DIR

COHORTS = {
    "PUB_KIRC_ICI_combined": INTERIM_PT / "PUB_KIRC_ICI_combined.adata.zarr",
    "NSCLC_Tissue_ICI_Pred": INTERIM_PT / "NSCLC_Tissue_ICI_Pred.adata.zarr",
    "PUB_BLCA_Mariathasan_EGAS00001002556": INTERIM_PT / "PUB_BLCA_Mariathasan_EGAS00001002556.adata.zarr",
    "PUB_ccRCC_Immotion150_and_151_ICI": INTERIM_PT / "PUB_ccRCC_Immotion150_and_151_ICI.adata.zarr",
    "PUB_ccRCC_Immotion150_and_151_TKI": INTERIM_PT / "PUB_ccRCC_Immotion150_and_151_TKI.adata.zarr",
}

MULTI_BATCH = ["PUB_KIRC_ICI_combined", "NSCLC_Tissue_ICI_Pred"]

ADV2_CONFIGS = {
    "Adv2_adversarial": CVAEAdv2Config(loss_type="adversarial", lambda_adv=1.0, lambda_mmd=0.0),
    "Adv2_mmd":         CVAEAdv2Config(loss_type="mmd",         lambda_adv=0.0, lambda_mmd=1.0),
    "Adv2_both":        CVAEAdv2Config(loss_type="both",        lambda_adv=1.0, lambda_mmd=0.5),
}

KL_CONFIG = CVAEConfig(latent_dim=128, normalize=True, n_epochs=150)

logger.info("Config ready. Multi-batch cohorts: %s", MULTI_BATCH)

## 1. Load Data

In [ ]:
all_adata = {}
for name, path in COHORTS.items():
    all_adata[name] = load_cohort(path)
    logger.info("  %s -> %d samples, %d batches, %d diagnoses",
                name, all_adata[name].n_obs,
                all_adata[name].obs[BATCH_COL].nunique(),
                all_adata[name].obs[DIAGNOSIS_COL].nunique())

## 2. Helper Functions

In [ ]:
def run_correction(adata, emb_key, batch_col, corrector):
    X = np.asarray(adata.obsm[emb_key]).astype(np.float32)
    batches = adata.obs[batch_col].astype(str).values
    corrected = corrector.fit_transform(X, batches)
    return corrected, corrector

def compute_all_metrics(adata, emb_key, batch_col, bio_col):
    n_batch = adata.obs[batch_col].nunique()
    n_bio = adata.obs[bio_col].nunique()
    m = {}

    if n_batch >= 2:
        try: m["kBET"] = compute_kbet(adata, emb_key, batch_col)
        except Exception as e: m["kBET"] = np.nan
        try: m["iLISI"] = compute_ilisi(adata, emb_key, batch_col)
        except Exception as e: m["iLISI"] = np.nan
        try: m["ASW_batch"] = compute_asw_batch(adata, emb_key, batch_col)
        except Exception as e: m["ASW_batch"] = np.nan
        try: m["graph_conn"] = compute_graph_connectivity(adata, batch_col, emb_key=emb_key)
        except Exception as e: m["graph_conn"] = np.nan
    else:
        m["kBET"] = m["iLISI"] = m["ASW_batch"] = m["graph_conn"] = np.nan

    if n_bio >= 2:
        try: m["cLISI"] = compute_clisi(adata, emb_key, bio_col)
        except Exception as e: m["cLISI"] = np.nan
        try: m["Sil_bio"] = compute_silhouette_bio(adata, emb_key, bio_col)
        except Exception as e: m["Sil_bio"] = np.nan
        try:
            adata_c = run_leiden_clustering(adata.copy(), emb_key, resolution=1.0)
            m["NMI"] = compute_nmi(adata_c, bio_col)
            m["ARI"] = compute_ari(adata_c, bio_col)
        except Exception as e:
            m["NMI"] = m["ARI"] = np.nan
    else:
        m["cLISI"] = m["Sil_bio"] = m["NMI"] = m["ARI"] = np.nan

    batch_vals = [m[k] for k in ["kBET","iLISI","ASW_batch","graph_conn"] if not np.isnan(m.get(k, np.nan))]
    bio_vals = [m[k] for k in ["cLISI","Sil_bio","NMI","ARI"] if not np.isnan(m.get(k, np.nan))]
    m["AvgBATCH"] = geometric_mean(batch_vals) if batch_vals else np.nan
    m["AvgBio"] = geometric_mean(bio_vals) if bio_vals else np.nan
    return m

def simple_sil(emb, labels):
    u = np.unique(labels)
    if len(u) < 2: return np.nan
    return silhouette_score(emb, labels)

logger.info("Helpers defined")

## 3. Run Batch Correction

Processing multi-batch cohorts (KIRC=5, NSCLC=8 batches) with:
1. **KL-cVAE** (baseline)
2. **Adv2 adversarial** (two-optimizer, no MMD)
3. **Adv2 mmd** (MMD only, no adversarial)
4. **Adv2 both** (adversarial + MMD)

In [ ]:
all_results = {}
all_histories = {}
all_correctors = {}

for cohort_name in MULTI_BATCH:
    logger.info("\n" + "=" * 60 + "\nProcessing: %s\n" + "=" * 60, cohort_name)
    adata = all_adata[cohort_name]
    X_raw = np.asarray(adata.obsm[EMB_KEY]).astype(np.float32)
    batches = adata.obs[BATCH_COL].astype(str).values

    results = {"Raw": X_raw}
    histories = {}
    correctors = {}

    # KL-cVAE baseline
    logger.info("--- KL-cVAE ---")
    kl_corr = CVAECorrector(KL_CONFIG)
    results["KL-cVAE"] = kl_corr.fit_transform(X_raw, batches)
    histories["KL-cVAE"] = kl_corr.history_
    correctors["KL-cVAE"] = kl_corr

    # Three Adv2 variants
    for variant_name, cfg in ADV2_CONFIGS.items():
        logger.info("--- %s ---", variant_name)
        corr = CVAEAdv2Corrector(cfg)
        results[variant_name] = corr.fit_transform(X_raw, batches)
        histories[variant_name] = corr.history_
        correctors[variant_name] = corr

    all_results[cohort_name] = results
    all_histories[cohort_name] = histories
    all_correctors[cohort_name] = correctors

logger.info("All corrections complete!")

## 4. Training Curves

In [ ]:
for cohort_name in MULTI_BATCH:
    histories = all_histories[cohort_name]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Training Curves: {cohort_name}", fontsize=14, fontweight="bold")

    for i, (method, hist) in enumerate(histories.items()):
        color = f"C{i}"
        axes[0, 0].plot(hist.recon, label=method, color=color, alpha=0.8)
        axes[0, 0].set_title("Reconstruction (MSE)")

        if hasattr(hist, "disc_accuracy") and hist.disc_accuracy:
            axes[0, 1].plot(hist.disc_accuracy, label=method, color=color, alpha=0.8)
            axes[0, 1].set_title("Disc Accuracy")
            n_b = all_adata[cohort_name].obs[BATCH_COL].nunique()
            if i == 0:
                axes[0, 1].axhline(1.0/n_b, ls="--", color="gray", alpha=0.5, label="random")

        if hasattr(hist, "adv_enc") and hist.adv_enc:
            axes[0, 2].plot(hist.adv_enc, label=method, color=color, alpha=0.8)
            axes[0, 2].set_title("Encoder Adv Loss")

        if hasattr(hist, "adv_disc") and hist.adv_disc:
            axes[1, 0].plot(hist.adv_disc, label=method, color=color, alpha=0.8)
            axes[1, 0].set_title("Disc CE Loss")

        if hasattr(hist, "mmd") and hist.mmd:
            axes[1, 1].plot(hist.mmd, label=method, color=color, alpha=0.8)
            axes[1, 1].set_title("MMD Loss")

        if hasattr(hist, "lambda_schedule") and hist.lambda_schedule:
            axes[1, 2].plot(hist.lambda_schedule, label=method, color=color, alpha=0.8)
            axes[1, 2].set_title("Lambda Schedule")

    for ax in axes.flat:
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 5. Before / After UMAP

In [ ]:
for cohort_name in MULTI_BATCH:
    results = all_results[cohort_name]
    adata = all_adata[cohort_name]
    methods = list(results.keys())
    n_methods = len(methods)

    fig, axes = plt.subplots(2, n_methods, figsize=(5 * n_methods, 10))
    fig.suptitle(f"UMAP: {cohort_name}", fontsize=14, fontweight="bold")

    for j, method in enumerate(methods):
        emb = results[method]
        tmp = ad.AnnData(obs=adata.obs.copy())
        tmp.obsm["X_emb"] = emb
        sc.pp.neighbors(tmp, use_rep="X_emb", n_neighbors=30)
        sc.tl.umap(tmp)
        umap_coords = tmp.obsm["X_umap"]

        batches = adata.obs[BATCH_COL].values
        for b in np.unique(batches):
            mask = batches == b
            axes[0, j].scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                              s=5, alpha=0.5, label=str(b)[:15])
        axes[0, j].set_title(f"{method}\n(Batch)")
        axes[0, j].legend(fontsize=5, loc="best", markerscale=2)

        diags = adata.obs[DIAGNOSIS_COL].values
        for d in np.unique(diags):
            mask = diags == d
            axes[1, j].scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                              s=5, alpha=0.5, label=str(d))
        axes[1, j].set_title(f"{method}\n(Diagnosis)")
        axes[1, j].legend(fontsize=7, loc="best", markerscale=2)

    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

## 6. Quick Silhouette Summary

In [ ]:
rows = []
for cohort_name in MULTI_BATCH:
    results = all_results[cohort_name]
    adata = all_adata[cohort_name]
    batches = adata.obs[BATCH_COL].astype(str).values
    diags = adata.obs[DIAGNOSIS_COL].astype(str).values
    for method, emb in results.items():
        rows.append({
            "Cohort": cohort_name.split("_")[1],
            "Method": method,
            "Sil_Batch": simple_sil(emb, batches),
            "Sil_Diag": simple_sil(emb, diags),
        })
df_sil = pd.DataFrame(rows)
df_sil["Batch_D"] = df_sil.groupby("Cohort")["Sil_Batch"].transform(lambda x: x - x.iloc[0])
df_sil["Diag_D"] = df_sil.groupby("Cohort")["Sil_Diag"].transform(lambda x: x - x.iloc[0])
display(df_sil.round(4))

## 7. Full scIB Metrics

Metrics from [Luecken et al. 2022](https://www.biorxiv.org/content/10.1101/2023.04.30.538439v2):
- **Batch**: kBET, iLISI, ASW_batch, Graph Connectivity -> **AvgBATCH**
- **Bio**: cLISI, Silhouette_bio, NMI, ARI -> **AvgBio**

In [ ]:
all_metrics = {}
for cohort_name in MULTI_BATCH:
    results = all_results[cohort_name]
    adata = all_adata[cohort_name]
    for method, emb in results.items():
        key = f"{cohort_name[:4]}_{method}"
        tmp = ad.AnnData(obs=adata.obs.copy())
        tmp.obsm["X_emb"] = emb
        logger.info("Metrics: %s / %s", cohort_name, method)
        m = compute_all_metrics(tmp, "X_emb", BATCH_COL, DIAGNOSIS_COL)
        m["Cohort"] = cohort_name
        m["Method"] = method
        all_metrics[key] = m

df_metrics = pd.DataFrame(all_metrics).T
cols = ["Cohort","Method","kBET","iLISI","ASW_batch","graph_conn","AvgBATCH","cLISI","Sil_bio","NMI","ARI","AvgBio"]
df_metrics = df_metrics[[c for c in cols if c in df_metrics.columns]]
display(df_metrics.round(4))

## 8. Method Comparison per Cohort

In [ ]:
for cohort_name in MULTI_BATCH:
    short = cohort_name.split("_")[1]
    mask = df_metrics["Cohort"] == cohort_name
    df_c = df_metrics[mask].set_index("Method").drop(columns=["Cohort"])
    print(f"\n{'='*60}")
    print(f"  {short}: Method Comparison")
    print(f"{'='*60}")
    display(df_c.round(4))
    for col in df_c.columns:
        vals = df_c[col].dropna()
        if len(vals) > 0:
            best = vals.idxmax()
            print(f"  Best {col}: {best} ({vals[best]:.4f})")

## 9. Cross-Cohort Analysis (no more NaN!)

Merge all 5 cohorts, use cohort name as batch label.
All metrics become computable since we have 5 'batches' and multiple diagnoses.

In [ ]:
all_raw = []
for name, adata in all_adata.items():
    tmp = ad.AnnData(obs=adata.obs.copy())
    tmp.obsm["X_raw"] = np.asarray(adata.obsm[EMB_KEY]).astype(np.float32)
    tmp.obs["cohort_batch"] = name
    all_raw.append(tmp)
merged = ad.concat(all_raw, join="outer")
logger.info("Merged: %d samples, %d cohort-batches",
            merged.n_obs, merged.obs["cohort_batch"].nunique())

# Raw
m_raw = compute_all_metrics(merged, "X_raw", "cohort_batch", DIAGNOSIS_COL)
m_raw["Method"] = "Raw"

# KL-cVAE on merged
X_merged = np.asarray(merged.obsm["X_raw"])
batches_m = merged.obs["cohort_batch"].astype(str).values

kl_m = CVAECorrector(KL_CONFIG)
merged.obsm["X_kl"] = kl_m.fit_transform(X_merged, batches_m)
m_kl = compute_all_metrics(merged, "X_kl", "cohort_batch", DIAGNOSIS_COL)
m_kl["Method"] = "KL-cVAE"

# Adv2 both on merged
adv2_m = CVAEAdv2Corrector(CVAEAdv2Config(loss_type="both"))
merged.obsm["X_adv2"] = adv2_m.fit_transform(X_merged, batches_m)
m_adv2 = compute_all_metrics(merged, "X_adv2", "cohort_batch", DIAGNOSIS_COL)
m_adv2["Method"] = "Adv2_both"

df_cross = pd.DataFrame({"Raw": m_raw, "KL-cVAE": m_kl, "Adv2_both": m_adv2}).T
print("\n" + "=" * 60)
print("  Cross-Cohort Comparison (5 cohorts merged)")
print("=" * 60)
display(df_cross.round(4))

## 10. Discriminator Convergence Diagnostic

In [ ]:
for cohort_name in MULTI_BATCH:
    histories = all_histories[cohort_name]
    n_b = all_adata[cohort_name].obs[BATCH_COL].nunique()
    random_bl = 1.0 / n_b

    fig, ax = plt.subplots(figsize=(10, 4))
    for method, hist in histories.items():
        if hasattr(hist, "disc_accuracy") and hist.disc_accuracy:
            ax.plot(hist.disc_accuracy, label=method, alpha=0.8)
    ax.axhline(random_bl, ls="--", color="red", alpha=0.7, label=f"Random ({random_bl:.2f})")
    ax.set_title(f"Disc Accuracy: {cohort_name} (n_batches={n_b})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)
    plt.tight_layout(); plt.show()

    for method, hist in histories.items():
        if hasattr(hist, "disc_accuracy") and hist.disc_accuracy:
            final = np.mean(hist.disc_accuracy[-10:])
            ok = "CONVERGED" if abs(final - random_bl) < 0.15 else "NOT converged"
            print(f"  {method}: final disc_acc={final:.3f} -> {ok}")

## Summary

### Expected outcomes:
- **disc_accuracy -> ~random** (1/n_batches) = adversarial is working
- **AvgBATCH up** = better batch mixing
- **AvgBio up or stable** = biology preserved
- **Adv2_both** should balance batch mixing and bio preservation better than KL-cVAE